In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
import random 

SEED = 21
random.seed(SEED)
np.random.seed(SEED)


In [3]:
import gzip

with gzip.open("data/train-images-idx3-ubyte.gz", "rb") as f:
    header = f.read(16)
    magic = np.frombuffer(header[0:4], dtype=">u4")[0]
    quantity = np.frombuffer(header[4:8], dtype=">u4")[0]
    height = np.frombuffer(header[8:12], dtype=">u4")[0]
    width = np.frombuffer(header[12:16], dtype=">u4")[0]
    print('Train massive Q =',quantity,'H =', height,'W =',
    width)

    big_buf = f.read()
    images_train = np.frombuffer(big_buf, dtype=np.uint8).reshape(quantity, height,
    width)

with gzip.open("data/train-labels-idx1-ubyte.gz", "rb") as f:
    header = f.read(8)
    labels_buf = f.read()
    labels_train = np.frombuffer(labels_buf, dtype="uint8")
    print (labels_train [:10])

with gzip.open("data/t10k-images-idx3-ubyte.gz", "rb") as f:
    header = f.read(16)
    magic = np.frombuffer(header[0:4], dtype=">u4")[0]
    quantity = np.frombuffer(header[4:8], dtype=">u4")[0]
    height = np.frombuffer(header[8:12], dtype=">u4")[0]
    width = np.frombuffer(header[12:16], dtype=">u4")[0]
    print('Test massive Q =',quantity,'H =', height,'W =',
    width)

    big_buf = f.read()
    images_test = np.frombuffer(big_buf, dtype=np.uint8).reshape(quantity, height,
    width)

with gzip.open("data/t10k-labels-idx1-ubyte.gz", "rb") as f:
    header = f.read(8)
    labels_buf = f.read()
    labels_test = np.frombuffer(labels_buf, dtype="uint8")
    print (labels_test [:10])

#NORMALIZATION

x_train = images_train.reshape(60000, 784)
x_train = x_train.astype(np.float32) / 255.0

x_test = images_test.reshape(10000, 784)
x_test = x_test.astype(np.float32) / 255.0

def v_vector(labels_any, classes=10):
    result = np.zeros((labels_any.shape[0], classes), dtype = np.float32)
    for i in range (labels_any.shape[0]):
        label = labels_any[i]
        result[i,label] = 1.0
    return result

y_train = v_vector(labels_train, classes = 10)
y_test = v_vector(labels_test, classes = 10)

Train massive Q = 60000 H = 28 W = 28
[9 0 0 3 0 2 7 2 5 5]
Test massive Q = 10000 H = 28 W = 28
[9 2 1 1 6 1 4 6 5 7]


In [4]:
def sigmoid(z):
    function = 1 / (1+np.exp(-z))
    return function
def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh(z):
    return np.tanh(z)
def tanh_deriv(z):
    t = np.tanh(z)
    return 1 - t ** 2

def relu(z):
    return np.maximum(0,z)
def relu_deriv(z):
    return (z > 0).astype(z.dtype)

In [ ]:
def init_w(layers, w_method="xav", cons=0.01, std=0.01, b_method="zeros", rng=None):
    if rng is None:
        rng = np.random
    
    params = []
    for i in range(len(layers)-1):
        layer_in = layers[i]
        layer_out = layers[i+1]

        if w_method == "cons":
            W = np.full((layer_out, layer_in), cons, dtype = np.float32)
        elif w_method == "norm":
            W = rng.randn(layer_out, layer_in).astype(np.float32)*std
        elif w_method == "xav":
            std = (2/(layer_in + layer_out))**0.5
            W = rng.randn(layer_out,layer_in).astype(np.float32)*std
        elif w_method == "kaim":
            std = (2/layer_in)**0.5
            W = rng.randn(layer_out,layer_in).astype(np.float32)*std
        else:
            raise ValueError(f"Unknown w_method: {w_method}")

        if b_method == "zeros":
            b = np.zeros(layer_out, dtype = np.float32)
        elif b_method == "rand":
            b = rng.randn(layer_out).astype(np.float32)*0.01
        else:
            raise ValueError(f"Unknown b_method: {b_method}")
        
        params.append((W, b))
    return params



In [9]:
def softmax(z):
    z_max = np.max(z, axis=1, keepdims=True)
    z_norm = z - z_max
    P = np.exp(z_norm)/(np.exp(z_norm).sum(axis=1, keepdims=True))
    return P

def calc_loss (P, Y):
    fix_zero = 10**(-10)
    N = P.shape[0]
    log_P = np.log(P + fix_zero)
    obnul = Y*log_P
    loss = -np.sum(obnul)/N
    return loss

def grad_sl (z, Y):
    P = softmax(z)
    N = P.shape[0]
    return (P-Y)/N


In [17]:
class SETKA:
    def __init__(self, layers: list[int], activation: str = "sigmoid", weights_method: str = "cons",dtype=np.float32, seed: int | None = 21):
        self.layers = layers
        self.act = activation
        self.seed = seed
        self.rng = np.random.RandomState(seed)
        self.params = init_w(layers, w_method=weights_method, cons=0.01, std=0.01, b_method="zeros", rng=self.rng)
        self.gradW = [np.zeros_like(W) for (W,b) in self.params]
        self.gradb = [np.zeros_like(b) for (W,b) in self.params]
        self.cache = []
        
        self.dtype = dtype

    def forward(self, X):

        A = X

        self.cache = []
        for i, (W, b) in enumerate(self.params):

            A_prev = A
            Z = A_prev @ W.T + b

            if i< len(self.params) - 1:
                if self.act == "sigmoid":
                    A = sigmoid(Z)
                elif self.act == "tanh":
                    A = tanh(Z)
                elif self.act == "relu":
                    A = relu(Z)
            else: 
                A = Z

            self.cache.append((A_prev, Z))

        return A
    
    def backward (self, grad_los_out):

        dZ = grad_los_out

        for i in reversed(range(len(self.params))):
            W, b = self.params[i]
            A_prev, Z = self.cache[i]
            self.gradW[i] = dZ.T @ A_prev
            self.gradb[i] = dZ.sum(axis=0)

            if i == 0:
                continue
            elif i > 0:
                dA_prev = dZ @ W
                Z_prev = self.cache[i-1][1]

                if self.act == "sigmoid":
                    dZ = dA_prev * sigmoid_deriv(Z_prev)
                elif self.act == "tanh":
                    dZ = dA_prev * tanh_deriv(Z_prev)
                elif self.act == "relu":
                    dZ = dA_prev * relu_deriv(Z_prev)

    def zero_grad(self):
        self.gradW = [np.zeros_like(W) for (W,b) in self.params]
        self.gradb = [np.zeros_like(b) for (W,b) in self.params]  

    def parametrs(self):
        return self.params

    def predict(self, X):
        logits = self.forward(X)
        return np.argmax(logits, axis=1)


        


In [18]:
N_train = 50000

X_tr = x_train[:N_train]      
Y_tr = y_train[:N_train]       
X_val = x_train[N_train:]     
Y_val = y_train[N_train:]     

def train_model(net, x_train, y_train, X_val, Y_val,
                epochs=5, batch_size=32, lr=0.01):
        
    N = x_train.shape[0]
    history = {
        "train_loss": [],
        "val_loss":   [],
        "val_acc":    [],
    }
    
    for epoch in range(epochs):
        indices = np.random.permutation(N)
        X_shuf = x_train[indices]
        Y_shuf = y_train[indices]
        
        epoch_loss = 0.0
        n_batches = 0
        
        for start in range(0, N, batch_size):
            end = start + batch_size
            X_batch = X_shuf[start:end]
            Y_batch = Y_shuf[start:end]
            
            logits = net.forward(X_batch)
            veroyatnosti = softmax(logits)
            
            loss = calc_loss(veroyatnosti, Y_batch)
            epoch_loss += loss
            n_batches += 1
            
            dL = grad_sl(logits, Y_batch)
            net.backward(dL)

            #непосредственно обучение
            for i, (W, b) in enumerate(net.params):
                W -= lr * net.gradW[i]
                b -= lr * net.gradb[i]
            
           
            net.zero_grad()
        
    
        val_logits = net.forward(X_val)
        val_probs = softmax(val_logits)
        val_loss = calc_loss(val_probs, Y_val)
        
        val_preds = np.argmax(val_logits, axis=1)
        val_true  = np.argmax(Y_val, axis=1)
        val_acc = (val_preds == val_true).mean()
        
        avg_train_loss = epoch_loss / n_batches
        
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        print(f"Epoch {epoch+1}/{epochs} | "
              f"train_loss={avg_train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | "
              f"val_acc={val_acc:.4f}")
    
    return history

net = SETKA(
    layers=[784, 100, 50, 10],
    activation="relu",
    weights_method="kaim",
    seed=21,
)

history = train_model(
    net,
    x_train=X_tr, y_train=Y_tr,
    X_val=X_val,   Y_val=Y_val,
    epochs=5,
    batch_size=32,
    lr=0.01,
)


Epoch 1/5 | train_loss=0.7366 | val_loss=0.5480 | val_acc=0.8036
Epoch 2/5 | train_loss=0.4965 | val_loss=0.5161 | val_acc=0.8097
Epoch 3/5 | train_loss=0.4508 | val_loss=0.4539 | val_acc=0.8383
Epoch 4/5 | train_loss=0.4252 | val_loss=0.4319 | val_acc=0.8464
Epoch 5/5 | train_loss=0.4065 | val_loss=0.4441 | val_acc=0.8414
